# ValoStats — Notebook 2: análisis predictivo de jugador reciente

## 1. Tareas de aprendizaje

El proyecto se organiza en cuatro tareas principales:

| Tarea | Entrada | Variable objetivo | Salida esperada |
|---|---|---|---|
| Clasificación de rendimiento | Métricas por partida como ACS, K/D, ADR, KAST, DDA, HS%, resultado y rondas jugadas | Nivel de rendimiento | Bajo, Medio, Alto o Destacado |
| Clasificación de estilo | Variables como agresividad, precisión, impacto, soporte, eficiencia, entry power y consistencia | Tipo de jugador | Alto impacto, Apoyo táctico u Ofensivo consistente |
| Análisis de tendencia | Comparación entre partidas antiguas y recientes dentro de las últimas 20 partidas | Tendencia reciente | Riesgo de bajar, Estable, Progreso positivo o Subida probable |
| Jugadores similares | Vector resumen del jugador y base de referencia | Similitud competitiva | Referentes similares en lobbies de rango parecido |

## 3. Métricas competitivas utilizadas

Para que el análisis no se base solo en estadísticas generales, ValoStats incorpora métricas competitivas usadas comúnmente en Valorant.

| Métrica | Estado en el proyecto | Uso dentro del análisis |
|---|---|---|
| ACS | Implementada | Impacto general por ronda. Se usa en rendimiento, tendencia y comparación. |
| ADR | Implementada | Daño promedio por ronda. Mide impacto ofensivo más allá de kills. |
| K/D | Implementada | Eficiencia en duelos. |
| KAST | Implementada | Participación y consistencia por ronda. |
| Headshot % | Implementada | Precisión mecánica. |
| First kills | Implementada | Iniciativa e impacto en duelos iniciales. |
| First deaths | Implementada | Riesgo en duelos iniciales. |
| Entry success | Implementada | Se calcula con first kills y first deaths. |
| DDA | Implementada | Diferencia de daño. |
| TRS / Tracker Score | Implementada | Puntaje general de Tracker.gg usado como apoyo. |
| Multi kills | Implementada | Impacto múltiple en rondas. |
| Clutch rate | No implementada directamente | No se calcula como tasa real porque no se obtienen de forma consistente intentos de clutch y clutches ganados. |
| Utility impact | Aproximada | Se aproxima mediante asistencias, soporte, KAST y participación por ronda. |

Esta decisión permite trabajar con métricas reales disponibles y declarar como limitación aquello que no se puede extraer de forma consistente.

## 4. Configuración inicial

Este notebook puede ejecutarse de dos formas:

- Modo rápido: usa el archivo `data/recent_matches.csv` ya generado.
- Modo completo: ejecuta el scraper.

Para una presentación se recomienda usar el modo rápido, porque el scraping completo puede tardar varios minutos.

In [ ]:
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        has_src = (candidate / "src").exists()
        has_data = (candidate / "data").exists()

        if has_src and has_data:
            return candidate

    if (current.parent / "src").exists():
        return current.parent

    return current

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

print("Raíz del proyecto:", PROJECT_ROOT)

## 5. Ejecución del pipeline completo

El script `src/run_full_analysis.py` centraliza todo el flujo.

Si `RUN_SCRAPER = False`, se reutiliza `data/recent_matches.csv`.

Si `RUN_SCRAPER = True`, el sistema abre Tracker.gg, extrae las últimas partidas competitivas y luego ejecuta las predicciones.

In [ ]:
RIOT_ID = "PoloGB#LAS"

# Cambiar a True solo si se quiere ejecutar el scraper real.
RUN_SCRAPER = True

# Cambiar a True cuando se actualice data/rank_reference_matches.csv y se quiera reconstruir data/rank_reference_profiles.csv.
REFRESH_REFERENCE = False

command = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "run_full_analysis.py"),
    RIOT_ID,
]

if not RUN_SCRAPER:
    command.append("--skip-scraper")

if REFRESH_REFERENCE:
    command.append("--refresh-reference")

print("Comando a ejecutar:")
print(" ".join(command))

In [ ]:
run_pipeline = True

if run_pipeline:
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("El pipeline falló. Revisar salida anterior.")
else:
    print("Pipeline no ejecutado desde el notebook. Se usará el JSON final existente.")

## 6. Carga del análisis final

El archivo `final_player_analysis.json` unifica todas las salidas del sistema:

- Predicción de rendimiento.
- Predicción de estilo.
- Tendencia temporal.
- Jugadores similares.
- Recomendaciones.
- Predicción por partida.
- Metodología.

In [ ]:
final_path = PROJECT_ROOT / "outputs" / "recent_predictions" / "final_player_analysis.json"

if not final_path.exists():
    raise FileNotFoundError(
        f"No existe {final_path}. Ejecuta primero el pipeline o genera el análisis desde la página."
    )

with open(final_path, "r", encoding="utf-8") as file:
    analysis = json.load(file)

analysis.keys()

## 7. Resumen del jugador analizado

In [ ]:
player = analysis["player"]
summary = analysis["summary"]
prediction = analysis["prediction_summary"]

player_summary = pd.DataFrame([
    {
        "Riot ID": player.get("riot_id"),
        "Rango actual": player.get("current_rank"),
        "Partidas analizadas": summary.get("matches_analyzed"),
        "Winrate": summary.get("winrate"),
        "K/D reciente": summary.get("recent_kd"),
        "ACS reciente": summary.get("recent_acs"),
        "ADR reciente": summary.get("recent_adr"),
        "KAST reciente": summary.get("recent_kast"),
        "Rendimiento global": prediction.get("performance_level"),
        "Estilo principal": prediction.get("main_style"),
        "Estilo secundario": prediction.get("secondary_style"),
        "Tendencia": prediction.get("trend_status"),
    }
])

player_summary

## 8. Predicción de rendimiento

El rendimiento se clasifica por partida y también a nivel global.

Las clases usadas son:

- Bajo
- Medio
- Alto
- Destacado

In [ ]:
performance_distribution = pd.DataFrame(
    list(prediction["performance_distribution"].items()),
    columns=["Rendimiento", "Cantidad de partidas"]
)

performance_distribution

In [ ]:
print("Estado competitivo:")
print(prediction["competitive_status"])

## 9. Predicción de estilo de juego

El estilo de juego se predice usando los perfiles construidos en el Notebook 1.

Las clases usadas son:

- Alto impacto
- Apoyo táctico
- Ofensivo consistente

In [ ]:
style_distribution = pd.DataFrame(
    list(prediction["style_distribution"].items()),
    columns=["Estilo", "Cantidad de partidas"]
)

style_distribution

## 10. Evolución temporal del jugador

Para incorporar evolución temporal, el sistema no analiza las últimas 20 partidas como un solo bloque estático.

En cambio, divide las partidas en dos tramos:

- Partidas 11 a 20: tramo anterior.
- Partidas 1 a 10: tramo reciente.

Esto permite observar progreso, caída o estabilidad.

In [ ]:
temporal = analysis.get("temporal_evolution")

if temporal is None:
    # Fallback por si el JSON fue generado antes de agregar temporal_evolution.
    trend_path = PROJECT_ROOT / "outputs" / "recent_predictions" / "trend_predictions.json"
    with open(trend_path, "r", encoding="utf-8") as file:
        trend_payload = json.load(file)
    temporal = trend_payload["global_trend_prediction"]

previous_half = temporal["previous_half"]
recent_half = temporal["recent_half"]
deltas = temporal["deltas"]

temporal_table = pd.DataFrame([
    {
        "Métrica": "Winrate",
        "Partidas 11-20": previous_half["winrate"],
        "Partidas 1-10": recent_half["winrate"],
        "Cambio": deltas["winrate"],
    },
    {
        "Métrica": "ACS promedio",
        "Partidas 11-20": previous_half["avg_acs"],
        "Partidas 1-10": recent_half["avg_acs"],
        "Cambio": deltas["avg_acs"],
    },
    {
        "Métrica": "K/D promedio",
        "Partidas 11-20": previous_half["avg_kd"],
        "Partidas 1-10": recent_half["avg_kd"],
        "Cambio": deltas["avg_kd"],
    },
    {
        "Métrica": "KAST promedio",
        "Partidas 11-20": previous_half["avg_kast"],
        "Partidas 1-10": recent_half["avg_kast"],
        "Cambio": deltas["avg_kast"],
    },
    {
        "Métrica": "Estilo principal",
        "Partidas 11-20": previous_half["main_style"],
        "Partidas 1-10": recent_half["main_style"],
        "Cambio": f"{previous_half['main_style']} -> {recent_half['main_style']}",
    },
])

temporal_table

In [ ]:
print("Tendencia global:")
print(temporal["trend_status"])
print()
print("Explicación:")
print(temporal["trend_explanation"])

## 11. Sistema de recomendación con jugadores similares

Para formalizar el sistema de recomendación, se implementó una búsqueda de jugadores similares usando Nearest Neighbors y similitud coseno.

El procedimiento es:

1. El jugador analizado se resume como un vector de métricas recientes.
2. Se calcula su media de rango de lobby.
3. Se filtra la base de referencia para usar jugadores de lobbies similares.
4. Se aplica Nearest Neighbors con similitud coseno.
5. Se calculan brechas contra el grupo similar.
6. Se generan recomendaciones.

Esto permite que las recomendaciones no dependan solo de reglas fijas, sino de una comparación vectorial con jugadores de contexto competitivo parecido.

In [ ]:
rank_context = analysis["rank_context"]
similar_summary = analysis["similar_group_summary"]
gap_analysis = analysis["gap_analysis"]

pd.DataFrame([
    {
        "Media de lobby": rank_context.get("target_avg_team_rank_nearest"),
        "Grupo de lobby": rank_context.get("target_avg_team_rank_group"),
        "Filtro usado": rank_context.get("filter_info", {}).get("filter_type"),
        "Jugadores comparados": similar_summary.get("players_compared"),
        "Winrate grupo similar": similar_summary.get("avg_winrate"),
        "K/D grupo similar": similar_summary.get("avg_kd"),
        "ACS grupo similar": similar_summary.get("avg_acs"),
    }
])

In [ ]:
similar_players = pd.DataFrame(analysis["similar_players"])

similar_players[
    [
        "rank",
        "reference_riot_id",
        "current_rank_mode",
        "avg_team_rank_nearest",
        "winrate",
        "recent_kd",
        "recent_acs",
        "recent_adr",
        "recent_kast",
        "main_agent",
    ]
].head(10)

In [ ]:
gap_table = pd.DataFrame(
    list(gap_analysis.items()),
    columns=["Brecha", "Valor"]
)

gap_table

## 12. Recomendaciones generadas

Las recomendaciones se construyen a partir de:

- Rendimiento global.
- Estilo principal y secundario.
- Tendencia reciente.
- Brechas frente al grupo de jugadores similares.

In [ ]:
for index, recommendation in enumerate(analysis["recommendations"], start=1):
    print(f"{index}. {recommendation}")

## 13. Predicción por partida

Además de la predicción global, el sistema genera predicciones para cada partida.

Esto permite observar qué partidas fueron positivas, negativas o neutras, y qué estilo se detectó en cada una.

In [ ]:
matches_df = pd.DataFrame(analysis["matches"])

matches_df[
    [
        "match_number",
        "date",
        "map",
        "agent",
        "result",
        "acs",
        "kills",
        "deaths",
        "assists",
        "performance_prediction",
        "style_prediction",
        "trend_signal",
    ]
]

## 14. Metodología resumida del sistema

El JSON final también guarda un resumen metodológico para explicar el sistema.

In [ ]:
methodology = analysis["methodology"]

print("Problema:")
print(methodology["problem"])
print()

print("Entrada:")
print(methodology["input"])
print()

print("Lógica de comparación por rango:")
print(methodology["rank_reference_logic"])

In [ ]:
pd.DataFrame(methodology["tasks"])

## 15. Limitaciones

Es importante declarar las limitaciones del sistema:

- La base de referencia puede crecer para mejorar la comparación por rango.
- La evolución temporal se calcula sobre las últimas 20 partidas, no sobre toda la temporada.
- Clutch rate y utility impact no se miden directamente, sino que se dejan como mejoras futuras o aproximaciones.
- El scraping depende de la disponibilidad y estructura de Tracker.gg.

In [ ]:
for limitation in methodology["limitations"]:
    print("-", limitation)